# Доверительные интервалы

## Модуль 3: Точечные и интервальные оценки параметров

В этом ноутбуке мы рассмотрим методы построения доверительных интервалов для параметров распределений.

### Содержание:
1. Точечные и интервальные оценки
2. Доверительный интервал для среднего (известная дисперсия)
3. t-распределение и доверительный интервал (неизвестная дисперсия)
4. Доверительный интервал для пропорции
5. Выбор размера выборки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Библиотеки успешно загружены!')

## 1. Точечные и интервальные оценки

### Точечная оценка
**Точечная оценка** — это одно число, которое используется для оценки параметра генеральной совокупности.

**Примеры:**
- $\bar{X}$ — оценка $\mu$
- $s^2$ — оценка $\sigma^2$
- $\hat{p}$ — оценка $p$

### Интервальная оценка
**Интервальная оценка** — это интервал, который с заданной вероятностью содержит истинное значение параметра.

**Доверительный интервал** уровня $1-\alpha$:
$$P(\theta_L \leq \theta \leq \theta_U) = 1 - \alpha$$

где:
- $\theta$ — истинное значение параметра
- $\theta_L$ — нижняя граница
- $\theta_U$ — верхняя граница
- $1-\alpha$ — уровень доверия (обычно 90%, 95%, 99%)

**Интерпретация:** Если повторить эксперимент много раз, то $1-\alpha$ процент построенных интервалов будет содержать истинное значение параметра.

## 2. Доверительный интервал для среднего (известная дисперсия)

Если $X_1, X_2, ..., X_n \sim N(\mu, \sigma^2)$ и $\sigma$ известна, то:

$$\bar{X} \sim N\left(\mu, \frac{\sigma^2}{n}\right)$$

$$Z = \frac{\bar{X} - \mu}{\sigma / \sqrt{n}} \sim N(0, 1)$$

**Доверительный интервал:**
$$\bar{X} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$

где $z_{\alpha/2}$ — квантиль стандартного нормального распределения.

**Стандартные значения $z_{\alpha/2}$:**
- 90%: $z = 1.645$
- 95%: $z = 1.96$
- 99%: $z = 2.576$

In [ ]:
# Пример: Доверительный интервал с известной дисперсией
np.random.seed(42)

# Параметры генеральной совокупности
mu_true = 170  # см (средний рост)
sigma_true = 10  # см (известная дисперсия)

# Выборка
n = 50
sample = np.random.normal(mu_true, sigma_true, n)
x_bar = np.mean(sample)

# Доверительные интервалы для разных уровней доверия
confidence_levels = [0.90, 0.95, 0.99]
z_scores = [1.645, 1.96, 2.576]

print('Доверительный интервал для среднего (σ известна)')
print('=' * 60)
print(f'Выборка: n = {n}, X̄ = {x_bar:.2f}, σ = {sigma_true}')
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Визуализация ДИ
for i, (cl, z) in enumerate(zip(confidence_levels, z_scores)):
    se = sigma_true / np.sqrt(n)
    margin = z * se
    ci_lower = x_bar - margin
    ci_upper = x_bar + margin
    
    print(f'{cl*100:.0f}% ДИ: [{ci_lower:.2f}, {ci_upper:.2f}], ширина = {2*margin:.2f}')
    
    axes[0].errorbar(x=i, y=x_bar, yerr=margin, fmt='o', markersize=10, 
                     capsize=10, capthick=2, label=f'{cl*100:.0f}% ДИ')

axes[0].axhline(y=mu_true, color='red', linestyle='--', linewidth=2, 
                label=f'Истинное μ = {mu_true}')
axes[0].axhline(y=x_bar, color='green', linestyle='-', linewidth=2, 
                label=f'X̄ = {x_bar:.2f}')
axes[0].set_ylabel('Рост (см)')
axes[0].set_title('Доверительные интервалы разного уровня')
axes[0].set_xticks(range(len(confidence_levels)))
axes[0].set_xticklabels([f'{cl*100:.0f}%' for cl in confidence_levels])
axes[0].legend()

# Симуляция: сколько раз истинное μ попадает в ДИ
n_simulations = 1000
n_in_ci = 0

for _ in range(n_simulations):
    sample = np.random.normal(mu_true, sigma_true, n)
    x_bar_sim = np.mean(sample)
    ci_lower = x_bar_sim - 1.96 * sigma_true / np.sqrt(n)
    ci_upper = x_bar_sim + 1.96 * sigma_true / np.sqrt(n)
    
    if ci_lower <= mu_true <= ci_upper:
        n_in_ci += 1

coverage = n_in_ci / n_simulations
print(f'\nСимуляция ({n_simulations} экспериментов):')
print(f'  Истинное μ попало в 95% ДИ: {n_in_ci}/{n_simulations} = {coverage:.3f}')
print(f'  Теоретическое покрытие: 0.950')

# Визуализация симуляции
x_range = np.linspace(mu_true - 4*sigma_true/np.sqrt(n), 
                       mu_true + 4*sigma_true/np.sqrt(n), 100)
axes[1].plot(x_range, stats.norm.pdf(x_range, mu_true, sigma_true/np.sqrt(n)), 
             'b-', linewidth=2, label='Распределение X̄')
axes[1].axvline(mu_true, color='red', linestyle='--', linewidth=2, label=f'μ = {mu_true}')
axes[1].fill_between(x_range, 0, 
                      stats.norm.pdf(x_range, mu_true, sigma_true/np.sqrt(n)), 
                      where=(x_range >= mu_true - 1.96*sigma_true/np.sqrt(n)) & 
                             (x_range <= mu_true + 1.96*sigma_true/np.sqrt(n)),
                      alpha=0.3, color='blue', label='95% ДИ')
axes[1].set_xlabel('X̄')
axes[1].set_ylabel('Плотность')
axes[1].set_title('Распределение выборочного среднего')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. t-распределение и доверительный интервал (неизвестная дисперсия)

На практике $\sigma$ обычно неизвестна, и мы используем выборочную дисперсию $s^2$.

**t-статистика:**
$$t = \frac{\bar{X} - \mu}{s / \sqrt{n}} \sim t(n-1)$$

где $t(n-1)$ — распределение Стьюдента с $n-1$ степенями свободы.

**Доверительный интервал:**
$$\bar{X} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

**Свойства t-распределения:**
- Похоже на нормальное, но с более тяжёлыми хвостами
- При $n \to \infty$ стремится к нормальному
- Чем меньше $n$, тем шире хвосты

In [ ]:
# Сравнение нормального и t-распределения
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PDF
x = np.linspace(-4, 4, 1000)
axes[0].plot(x, stats.norm.pdf(x), 'b-', linewidth=2, label='N(0,1)')

df_values = [1, 3, 5, 10, 30]
colors = ['red', 'green', 'orange', 'purple', 'brown']

for df, color in zip(df_values, colors):
    axes[0].plot(x, stats.t.pdf(x, df), '--', color=color, linewidth=2, 
                 label=f't({df})')

axes[0].set_xlabel('x')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Нормальное и t-распределения')
axes[0].legend()
axes[0].set_ylim(0, 0.45)

# Разница в квантилях
alpha = 0.05
n_values = np.arange(2, 101)
t_quantiles = [stats.t.ppf(1 - alpha/2, n-1) for n in n_values]
z_quantile = stats.norm.ppf(1 - alpha/2)

axes[1].plot(n_values, t_quantiles, 'b-', linewidth=2, label='t-квантиль')
axes[1].axhline(y=z_quantile, color='red', linestyle='--', linewidth=2, 
                label=f'z-квантиль = {z_quantile:.3f}')
axes[1].set_xlabel('Размер выборки (n)')
axes[1].set_ylabel('Квантиль')
axes[1].set_title('Сходимость t-квантиля к z-квантилю')
axes[1].legend()
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Пример: Доверительный интервал с неизвестной дисперсией
np.random.seed(42)

# Данные: время доставки заказов (минуты)
delivery_times = np.array([25, 30, 28, 35, 32, 27, 29, 31, 26, 33,
                           28, 30, 34, 27, 29, 31, 26, 32, 28, 30])

n = len(delivery_times)
x_bar = np.mean(delivery_times)
s = np.std(delivery_times, ddof=1)
se = s / np.sqrt(n)

# t-квантиль для 95% ДИ
alpha = 0.05
t_critical = stats.t.ppf(1 - alpha/2, n-1)

# Доверительный интервал
ci_lower = x_bar - t_critical * se
ci_upper = x_bar + t_critical * se

print('Доверительный интервал для среднего (σ неизвестна)')
print('=' * 60)
print(f'Данные: время доставки (n = {n})')
print(f'X̄ = {x_bar:.2f} мин')
print(f's = {s:.2f} мин')
print(f'SE = s/√n = {se:.2f} мин')
print(f't-квантиль (α/2={alpha/2}, df={n-1}): {t_critical:.3f}')
print(f'\n95% ДИ: [{ci_lower:.2f}, {ci_upper:.2f}] мин')
print(f'Ширина ДИ: {ci_upper - ci_lower:.2f} мин')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма данных
axes[0].hist(delivery_times, bins=10, density=True, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(x_bar, color='red', linestyle='--', linewidth=2, label=f'X̄ = {x_bar:.1f}')
axes[0].set_xlabel('Время доставки (мин)')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Распределение времени доставки')
axes[0].legend()

# Доверительный интервал
axes[1].errorbar(x=0, y=x_bar, yerr=t_critical * se, 
                 fmt='o', markersize=10, capsize=10, capthick=2, color='blue',
                 label='95% ДИ')
axes[1].axhline(y=x_bar, color='green', linestyle='-', linewidth=2, 
                label=f'X̄ = {x_bar:.2f}')
axes[1].set_ylabel('Время доставки (мин)')
axes[1].set_title('95% доверительный интервал')
axes[1].legend()
axes[1].set_xlim(-0.5, 0.5)
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

## 4. Доверительный интервал для пропорции

Для бинарных данных (успех/неудача) используется пропорция.

**Выборочная пропорция:**
$$\hat{p} = \frac{X}{n}$$

где $X$ — количество успехов в $n$ испытаниях.

**Распределение:**
$$\hat{p} \sim N\left(p, \frac{p(1-p)}{n}\right)$$

**Доверительный интервал (Вальд):**
$$\hat{p} \pm z_{\alpha/2} \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

**Условие применимости:**
$n\hat{p} \geq 5$ и $n(1-\hat{p}) \geq 5$

In [ ]:
# Пример: Доверительный интервал для пропорции
np.random.seed(42)

# Данные: опрос покупателей
n = 200  # Количество опрошенных
x = 124  # Количество положительных отзывов

p_hat = x / n

# Доверительный интервал (Вальд)
alpha = 0.05
z = stats.norm.ppf(1 - alpha/2)
se = np.sqrt(p_hat * (1 - p_hat) / n)
ci_lower = p_hat - z * se
ci_upper = p_hat + z * se

print('Доверительный интервал для пропорции')
print('=' * 60)
print(f'Данные: {x} положительных из {n} опрошенных')
print(f'p̂ = {p_hat:.4f} ({p_hat*100:.1f}%)')
print(f'SE = √(p̂(1-p̂)/n) = {se:.4f}')
print(f'z = {z:.3f}')
print(f'\n95% ДИ: [{ci_lower:.4f}, {ci_upper:.4f}]')
print(f'         [{ci_lower*100:.1f}%, {ci_upper*100:.1f}%]')

# Условие применимости
print(f'\nУсловие Вальда:')
print(f'  np̂ = {n*p_hat:.1f} ≥ 5? {n*p_hat >= 5}')
print(f'  n(1-p̂) = {n*(1-p_hat):.1f} ≥ 5? {n*(1-p_hat) >= 5}')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Распределение p̂
x_range = np.linspace(p_hat - 4*se, p_hat + 4*se, 1000)
axes[0].plot(x_range, stats.norm.pdf(x_range, p_hat, se), 'b-', linewidth=2)
axes[0].fill_between(x_range, stats.norm.pdf(x_range, p_hat, se),
                     where=(x_range >= ci_lower) & (x_range <= ci_upper),
                     alpha=0.3, color='blue', label='95% ДИ')
axes[0].axvline(p_hat, color='red', linestyle='--', linewidth=2, label=f'p̂ = {p_hat:.3f}')
axes[0].set_xlabel('p')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Распределение выборочной пропорции')
axes[0].legend()

# Доверительный интервал
axes[1].errorbar(x=0, y=p_hat, yerr=z * se, 
                 fmt='o', markersize=10, capsize=10, capthick=2, color='blue',
                 label='95% ДИ')
axes[1].axhline(y=p_hat, color='green', linestyle='-', linewidth=2, 
                label=f'p̂ = {p_hat:.3f}')
axes[1].set_ylabel('Пропорция')
axes[1].set_title('95% доверительный интервал')
axes[1].legend()
axes[1].set_xlim(-0.5, 0.5)
axes[1].set_xticks([])
axes[1].set_ylim(0.5, 0.75)

plt.tight_layout()
plt.show()

## 5. Выбор размера выборки

### Для оценки среднего:
$$n = \left(\frac{z_{\alpha/2} \cdot \sigma}{E}\right)^2$$

где $E$ — желаемая точность (половина ширины ДИ).

### Для оценки пропорции:
$$n = \frac{z_{\alpha/2}^2 \cdot p(1-p)}{E^2}$$

**Примечание:** Если $p$ неизвестно, используйте $p = 0.5$ (максимальная дисперсия).

In [ ]:
# Пример: Выбор размера выборки

# Для оценки среднего
sigma = 15  # Предполагаемое стандартное отклонение
E_values = [1, 2, 3, 5]  # Желаемая точность
confidence_levels = [0.90, 0.95, 0.99]

print('Выбор размера выборки для оценки среднего')
print('=' * 60)
print(f'σ = {sigma}')
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cl in confidence_levels:
    z = stats.norm.ppf(1 - (1-cl)/2)
    n_values = []
    for E in E_values:
        n = (z * sigma / E) ** 2
        n_values.append(np.ceil(n))
    
    axes[0].plot(E_values, n_values, 'o-', linewidth=2, markersize=8, 
                 label=f'{cl*100:.0f}% ДИ')

axes[0].set_xlabel('Желаемая точность (E)')
axes[0].set_ylabel('Размер выборки (n)')
axes[0].set_title('Размер выборки vs точность\n(для оценки среднего)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Для оценки пропорции
p_values = [0.1, 0.3, 0.5, 0.7, 0.9]
E = 0.05  # Точность 5%

print(f'\nВыбор размера выборки для оценки пропорции (E = {E})')
print('=' * 60)

for cl in confidence_levels:
    z = stats.norm.ppf(1 - (1-cl)/2)
    n_values = []
    for p in p_values:
        n = (z**2 * p * (1-p)) / E**2
        n_values.append(np.ceil(n))
    
    axes[1].plot(p_values, n_values, 'o-', linewidth=2, markersize=8, 
                 label=f'{cl*100:.0f}% ДИ')

axes[1].set_xlabel('Ожидаемая пропорция (p)')
axes[1].set_ylabel('Размер выборки (n)')
axes[1].set_title(f'Размер выборки vs пропорция\n(E = {E})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Пример расчёта
print('\nПример расчёта:')
print(f'Для σ = {sigma}, E = 2, 95% ДИ:')
z = stats.norm.ppf(0.975)
n = (z * sigma / 2) ** 2
print(f'  n = (z * σ / E)² = ({z:.3f} * {sigma} / 2)² = {n:.1f} → {int(np.ceil(n))}')

## Упражнения

### Упражнение 1: Доверительный интервал для среднего
Даны результаты экзамена (баллы):
```
[72, 85, 90, 68, 75, 82, 88, 92, 78, 80, 85, 90, 76, 83, 87]
```

1. Постройте 95% доверительный интервал для среднего балла
2. Постройте 99% доверительный интервал
3. Как изменится ширина ДИ при увеличении выборки в 4 раза?

### Упражнение 2: Доверительный интервал для пропорции
В опросе 500 человек 320 поддержали новую политику.
1. Постройте 95% доверительный интервал для доли сторонников
2. Какой минимальный размер выборки нужен для точности ±3%?

### Упражнение 3: Выбор размера выборки
Магазин хочет оценить средний чек с точностью ±10 руб. (σ ≈ 50 руб.).
1. Какой размер выборки нужен для 95% ДИ?
2. Для 99% ДИ?
3. Как изменится результат, если σ = 100 руб.?

### Упражнение 4: Интерпретация ДИ
Пусть 95% доверительный интервал для среднего времени доставки: [28.5, 31.5] мин.
1. Верно ли утверждение: "С вероятностью 95% истинное среднее лежит в этом интервале"?
2. Как правильно интерпретировать этот ДИ?

---

**Решения** можно найти в ноутбуке `solutions/07_Solutions.ipynb`